# OceanGuard AI — Marine Debris Intelligence with Gemma 4 E2B + Tool Calling

**Reproducibility companion for Kaggle Gemma 4 Good Hackathon submission**

Track: Global Resilience

Author: Alejandro Sanchez Ferrer (`asanc.tech@gmail.com`)

---

This notebook reproduces the core intelligence pipeline of the **OceanGuard AI** Android app in a
standalone Python environment. The app (`github.com/asferrer/OceanguardAI-App`) runs entirely
offline on a Samsung Galaxy S22 Ultra (Exynos 2200, no GPU delegate). The notebook uses the same
two-phase tool-calling architecture but swaps Android-specific runtimes (LiteRT-LM, TFLite XNNPACK)
for HuggingFace Transformers + bitsandbytes 4-bit quantization, making it fully reproducible on
a Kaggle T4 instance.

## Problem Statement

Marine plastic pollution is one of the most critical environmental crises of our time: an estimated
8–12 million metric tons of plastic enter the ocean each year, threatening over 800 marine species
through entanglement, ingestion, and chemical leaching. Coastal cleanup campaigns remain largely
manual and data-sparse — field teams rarely have access to systematic monitoring tools in remote
or low-connectivity environments.

Existing detection tools either require cloud connectivity (breaking offline field scenarios) or
run on desktop/server hardware far removed from the collection site. The gap between fast
real-time detection and deep analytical reporting has never been closed on consumer mobile hardware,
leaving citizen scientists without actionable, grounded intelligence at the point of collection.

OceanGuard AI addresses this with a **fully-offline dual-pipeline approach** on Android: RT-DETRv2
for fast frame-level detection and Gemma 4 E2B for deep analysis and natural-language report
generation, connected by a native tool-calling bridge that forces the VLM to query pre-computed
statistics before writing any prose. This eliminates the hallucinated percentages and fabricated
debris classes that plague naive prompt-and-generate pipelines, producing scientifically grounded
reports in six languages directly on-device.

In [ ]:
%pip install -q "transformers>=4.51" bitsandbytes accelerate pillow torch

## 1. Setup & Dependencies

In [ ]:
"""Imports, GPU diagnostics, and reproducibility seed."""
from __future__ import annotations

import json
import random
import re
import unicodedata
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any

import torch
from PIL import Image
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# GPU diagnostics
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {device_name}  |  VRAM: {total_vram_gb:.1f} GB")
else:
    print("No CUDA GPU detected — running on CPU (expect slow inference)")

print(f"torch {torch.__version__}")

## 2. Load Gemma 4 E2B (4-bit Quantized)

In [ ]:
"""Load Gemma 4 E2B with bitsandbytes 4-bit quantization."""

# TODO: confirm exact HuggingFace tag once Gemma 4 E2B is publicly released
# Candidate tags: "google/gemma-4-E2B-it", "google/gemma-4-2b-it"
# The app uses google/gemma-3n-E2B-it-int4 (LiteRT-LM format, 3.4 GB)
# For Kaggle we use the HF checkpoint with bitsandbytes 4-bit to match memory footprint
MODEL_ID = "google/gemma-4-E2B-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print(f"Loading tokenizer from {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print(f"Loading model {MODEL_ID} with 4-bit quantization ...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
)
model.eval()

if torch.cuda.is_available():
    allocated_gb = torch.cuda.memory_allocated() / 1e9
    reserved_gb = torch.cuda.memory_reserved() / 1e9
    print(f"GPU memory — allocated: {allocated_gb:.2f} GB  |  reserved: {reserved_gb:.2f} GB")

print("Model ready.")

## 3. Detector — Debris Detection (RT-DETRv2 Inspired)

The Android app runs **RT-DETRv2** as a TFLite model (`rtdetrv2_detector.tflite`, 82.8 MB FP16)
via XNNPACK with 8 CPU threads on the Exynos 2200. The model performs end-to-end detection at
640x640 and outputs bounding boxes in normalized `[0,1]` coordinates.

Porting that model to Kaggle is non-trivial: the ONNX surgery that replaces `Erf` with a tanh
approximation (erf-free variant) is Android-specific, and the full conversion pipeline lives in
`conversion/` of the app repo. Rather than blocking the demonstration on a complex conversion,
this cell defines a **mock detector** that returns realistic synthetic detections.
The real inference code is at:
`android/app/src/main/java/com/oceanguard/inference/RTDETRInference.kt`

In [ ]:
"""Mock RT-DETRv2 detector returning hardcoded synthetic detections."""

# The 8 classes the Android model was trained on (COCO-style UPPER_SNAKE_CASE)
DEBRIS_CLASSES = [
    "Bottle",
    "Can",
    "Fishing_Net",
    "Glove",
    "Mask",
    "Metal_Debris",
    "Plastic_Debris",
    "Tire",
]


@dataclass
class Detection:
    """Single debris detection with normalized bounding box."""

    label: str
    confidence: float
    # Normalized [0, 1] box: [x_min, y_min, x_max, y_max]
    box: list[float] = field(default_factory=lambda: [0.0, 0.0, 1.0, 1.0])
    material: str = "PLASTIC"  # inferred from class


# Material inference map (mirrors EnvironmentalImpact.getImpact in Android codebase)
_CLASS_TO_MATERIAL: dict[str, str] = {
    "Bottle": "PLASTIC",
    "Can": "METAL",
    "Fishing_Net": "NYLON",
    "Glove": "RUBBER",
    "Mask": "PLASTIC",
    "Metal_Debris": "METAL",
    "Plastic_Debris": "PLASTIC",
    "Tire": "RUBBER",
}


def detect_debris(image: Image.Image) -> list[Detection]:
    """Return synthetic detections for a single image (mock RT-DETRv2)."""
    # Hardcoded detections representative of a coastal cleanup scenario
    raw = [
        {"label": "Bottle",        "confidence": 0.92, "box": [0.10, 0.15, 0.35, 0.60]},
        {"label": "Plastic_Debris","confidence": 0.78, "box": [0.45, 0.20, 0.80, 0.75]},
        {"label": "Fishing_Net",   "confidence": 0.81, "box": [0.60, 0.05, 0.95, 0.50]},
    ]
    return [
        Detection(
            label=d["label"],
            confidence=d["confidence"],
            box=d["box"],
            material=_CLASS_TO_MATERIAL.get(d["label"], "UNKNOWN"),
        )
        for d in raw
    ]


# Smoke-test
_test_img = Image.new("RGB", (640, 640), color=(10, 30, 60))
_detections = detect_debris(_test_img)
for det in _detections:
    print(f"  {det.label:20s}  conf={det.confidence:.2f}  material={det.material}")

## 4. Tool Calling — Pre-Rendered Data Bundle

The Android app wires Gemma 4 to **7 no-argument tools** via LiteRT-LM native tool calling
(see `android/app/src/main/java/com/oceanguard/inference/OceanGuardTools.kt`).
Each tool returns pre-computed statistics from `ToolReportContext` rather than letting the
model fabricate numbers.

**Key design decision from the Android codebase (`ToolReportGenerator.kt`, lines 36-44):**
Tools are defined as no-argument to avoid the LiteRT-LM FC parser bug where string arguments
without surrounding quotes cause a parse rejection. The Python port mirrors this constraint
and uses HuggingFace chat template tool calling.

The four tools below are the notebook subset of the full seven; the complete set
(`get_debris_summary`, `get_material_breakdown`, `get_type_breakdown`, `get_risk_assessment`,
`get_collection_waypoints`, `get_ecological_impacts`, `get_survey_statistics`) is implemented
in cells 16+ (v2).

In [ ]:
"""Tool definitions and execution engine for the Gemma 4 tool-calling loop."""


@dataclass
class SurveyContext:
    """Pre-computed survey statistics, analogous to Android's ToolReportContext."""

    detections: list[Detection]
    image_path: str = ""
    gps_lat: float | None = None
    gps_lon: float | None = None
    timestamp: str = field(default_factory=lambda: datetime.utcnow().strftime("%Y-%m-%d"))

    @property
    def type_counts(self) -> dict[str, int]:
        counts: dict[str, int] = {}
        for d in self.detections:
            counts[d.label] = counts.get(d.label, 0) + 1
        return counts

    @property
    def material_counts(self) -> dict[str, int]:
        counts: dict[str, int] = {}
        for d in self.detections:
            counts[d.material] = counts.get(d.material, 0) + 1
        return counts

    @property
    def avg_confidence(self) -> float:
        if not self.detections:
            return 0.0
        return sum(d.confidence for d in self.detections) / len(self.detections)

    @property
    def health_score(self) -> int:
        """0-100 health score: 100 - (25 * count), floored at 0."""
        return max(0, 100 - 25 * len(self.detections))


# Tool schema definitions for Gemma 4 chat template
TOOL_SCHEMAS: list[dict[str, Any]] = [
    {
        "type": "function",
        "function": {
            "name": "get_total_debris_count",
            "description": (
                "Returns the total number of debris items detected in this survey. "
                "Call this first to establish the survey scale."
            ),
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_debris_by_category",
            "description": (
                "Returns count and percentage for each debris category detected. "
                "Returned list is authoritative — do not invent unlisted categories."
            ),
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_session_metadata",
            "description": (
                "Returns survey metadata: timestamp, GPS coordinates, health score, "
                "average detection confidence."
            ),
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_locations",
            "description": (
                "Returns GPS waypoints for prioritized debris collection. "
                "Returns empty list when GPS is unavailable."
            ),
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
]

REQUIRED_TOOLS = {s["function"]["name"] for s in TOOL_SCHEMAS}


def execute_tool(name: str, ctx: SurveyContext) -> Any:
    """Dispatch a tool call by name and return its result."""
    if name == "get_total_debris_count":
        return {"total": len(ctx.detections)}

    if name == "get_debris_by_category":
        total = max(len(ctx.detections), 1)
        items = [
            {
                "name": label,
                "count": count,
                "percent": round(count * 100.0 / total, 1),
            }
            for label, count in ctx.type_counts.items()
        ]
        return {"total": total, "items": items}

    if name == "get_session_metadata":
        return {
            "timestamp": ctx.timestamp,
            "gps_lat": ctx.gps_lat,
            "gps_lon": ctx.gps_lon,
            "health_score": ctx.health_score,
            "avg_confidence": round(ctx.avg_confidence, 3),
            "analyzed_images": 1,
        }

    if name == "get_locations":
        if ctx.gps_lat is None or ctx.gps_lon is None:
            return {"count": 0, "items": []}
        return {
            "count": 1,
            "items": [
                {
                    "lat": ctx.gps_lat,
                    "lon": ctx.gps_lon,
                    "debris_count": len(ctx.detections),
                    "health_score": ctx.health_score,
                    "priority": "HIGH" if ctx.health_score < 50 else "MEDIUM",
                }
            ],
        }

    raise ValueError(f"Unknown tool: {name}")


print(f"Registered tools: {sorted(REQUIRED_TOOLS)}")

## 5. Two-Phase Orchestration

The Android app uses a strict **two-phase protocol** (`ToolReportGenerator.kt`, lines 261-303):

- **PHASE 1**: Gemma 4 invokes all required tools in a conversation turn. Any prose output is
  discarded — the model is only allowed to emit tool calls. The LiteRT-LM `ToolAgentLoop`
  (`ToolAgentLoop.kt`, lines 115-118) exits as soon as all required tools have run, preventing
  KV-cache contamination from premature hallucinated drafts.

- **PHASE 2**: A **fresh conversation** receives only the pre-rendered markdown data bundle
  (ground-truth tables built from tool results). No tools are available. The model's single
  job is to copy tables verbatim and write connecting prose. This "copy these tables" framing
  dramatically reduces hallucination rate in Gemma 4 E2B.

The Python port implements the same pattern using HuggingFace `apply_chat_template` with
`tools=TOOL_SCHEMAS` for PHASE 1 and a plain system+user prompt for PHASE 2.

In [ ]:
"""Two-phase report orchestrator: tool dispatch + grounded narrative generation."""

MAX_TOOL_ROUNDS = 6      # mirrors ToolReportGenerator.MAX_TOOL_ROUNDS (Android uses 14)
MAX_NEW_TOKENS_P1 = 512  # PHASE 1 is tool calls only; capped low
MAX_NEW_TOKENS_P2 = 1024 # PHASE 2 is the full narrative


def _normalize_label(s: str) -> str:
    """Lowercase, collapse whitespace, strip combining diacritics."""
    decomposed = unicodedata.normalize("NFD", s.strip().lower())
    stripped = "".join(c for c in decomposed if unicodedata.category(c) != "Mn")
    return re.sub(r"\s+", " ", stripped)


def _build_data_bundle(tool_results: dict[str, Any], ctx: SurveyContext) -> str:
    """Render tool results as a markdown data bundle for PHASE 2."""
    lines: list[str] = [
        "## CONFIRMED DATA — USE THESE EXACT VALUES",
        "",
        f"Survey date: {ctx.timestamp}",
        f"Health score: {ctx.health_score}/100",
        "",
    ]

    total_result = tool_results.get("get_total_debris_count", {})
    lines += [
        f"**Total debris items detected: {total_result.get('total', 0)}**",
        "",
    ]

    cat_result = tool_results.get("get_debris_by_category", {})
    if cat_result.get("items"):
        lines += ["### Debris by Category", "", "| Category | Count | % |", "| --- | --- | --- |"]
        for item in cat_result["items"]:
            lines.append(f"| {item['name']} | {item['count']} | {item['percent']}% |")
        lines.append("")

    meta_result = tool_results.get("get_session_metadata", {})
    if meta_result:
        lines += [
            "### Survey Metadata",
            "",
            f"- Analyzed images: {meta_result.get('analyzed_images', 1)}",
            f"- Average detection confidence: {meta_result.get('avg_confidence', 0):.1%}",
        ]
        if meta_result.get("gps_lat") is not None:
            lines.append(
                f"- GPS: {meta_result['gps_lat']:.4f}, {meta_result['gps_lon']:.4f}"
            )
        lines.append("")

    loc_result = tool_results.get("get_locations", {})
    if loc_result.get("items"):
        lines += ["### Collection Waypoints", "", "| Lat | Lon | Count | Priority |", "| --- | --- | --- | --- |"]
        for w in loc_result["items"]:
            lines.append(f"| {w['lat']:.4f} | {w['lon']:.4f} | {w['debris_count']} | {w['priority']} |")
        lines.append("")

    return "\n".join(lines)


def _run_phase1(
    ctx: SurveyContext,
    language: str = "en",
) -> dict[str, Any]:
    """PHASE 1: model invokes tools; collect and execute all tool calls."""
    tool_results: dict[str, Any] = {}
    called_tools: set[str] = set()

    system_msg = (
        f"You are a marine debris analyst. Language: English.\n"
        "PHASE 1: Call each available tool once (all no-argument). "
        "Emit NO prose — only tool calls. "
        "Required tools: get_total_debris_count, get_debris_by_category, "
        "get_session_metadata, get_locations."
    )
    user_msg = (
        "Invoke all required tools now to gather survey data. No prose yet."
    )

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user",   "content": user_msg},
    ]

    remaining = set(REQUIRED_TOOLS)
    for _round in range(MAX_TOOL_ROUNDS):
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tools=TOOL_SCHEMAS,
            add_generation_prompt=True,
            tokenize=False,
        )
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS_P1,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        generated = tokenizer.decode(
            output_ids[0][inputs["input_ids"].shape[-1]:],
            skip_special_tokens=False,
        )
        print(f"  [PHASE 1 round {_round + 1}] raw output (first 200 chars): {generated[:200]!r}")

        # Parse tool calls from the generated text.
        # Gemma 4 emits tool calls as JSON wrapped in <tool_call>...</tool_call> tags.
        # TODO: adjust parsing once exact Gemma 4 output format is confirmed on HF
        tool_call_matches = re.findall(
            r"<tool_call>\s*(\{.*?\})\s*</tool_call>",
            generated,
            re.DOTALL,
        )

        if not tool_call_matches:
            # No tool calls emitted — check if all required tools already ran
            if not remaining:
                break
            # Inject redirect (mirrors ToolAgentLoop redirect logic, lines 136-141 in Kotlin)
            missing_list = ", ".join(sorted(remaining))
            redirect = (
                f"STOP. You have not yet invoked the required tools: {missing_list}. "
                "Invoke each of them now. Do NOT write any prose."
            )
            messages.append({"role": "assistant", "content": generated})
            messages.append({"role": "user", "content": redirect})
            continue

        # Execute each tool call and accumulate results
        tool_responses: list[dict[str, Any]] = []
        for raw_json in tool_call_matches:
            try:
                call = json.loads(raw_json)
                tool_name: str = call.get("name", "")
                result = execute_tool(tool_name, ctx)
                tool_results[tool_name] = result
                called_tools.add(tool_name)
                remaining.discard(tool_name)
                print(f"  Tool executed: {tool_name} -> {json.dumps(result)[:120]}")
                tool_responses.append({"role": "tool", "name": tool_name, "content": json.dumps(result)})
            except (json.JSONDecodeError, ValueError) as exc:
                print(f"  Tool parse error: {exc}")

        messages.append({"role": "assistant", "content": generated})
        messages.extend(tool_responses)

        if not remaining:
            print(f"  [PHASE 1] All required tools invoked after round {_round + 1}")
            break

    # Fallback: execute any tools the model skipped directly
    for tool_name in list(remaining):
        print(f"  [PHASE 1 fallback] Direct execution of skipped tool: {tool_name}")
        tool_results[tool_name] = execute_tool(tool_name, ctx)

    return tool_results


def _run_phase2(bundle: str, language: str = "en") -> str:
    """PHASE 2: fresh conversation, model writes narrative from data bundle."""
    writer_system = (
        "You are a senior marine conservation scientist authoring a field assessment. "
        "RULES:\n"
        "- CLOSED-WORLD: mention only debris types that appear in the CONFIRMED DATA tables.\n"
        "- COPY TABLES VERBATIM: reproduce each table with identical rows and numbers.\n"
        "- NO PLACEHOLDERS: never emit [anything], TODO, or unfilled brackets.\n"
        "- HUMAN LABELS: use readable labels, never UPPER_SNAKE_CASE identifiers.\n"
        "- TONE: scientific but accessible. 400-600 words of body prose.\n"
        "OUTPUT LANGUAGE: English."
    )

    messages = [
        {"role": "system", "content": writer_system},
        {"role": "user",   "content": bundle},
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS_P2,
            do_sample=True,
            temperature=0.3,  # matches Android app: temperature=0.3 in OceanGuardInference.kt
            top_k=20,          # matches Android app: topK=20
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )


def generate_grounded_report(
    image: Image.Image,
    language: str = "en",
    gps_lat: float | None = None,
    gps_lon: float | None = None,
) -> str:
    """Full two-phase pipeline: detect -> tool calling -> grounded narrative."""
    print("[DETECT] Running debris detector ...")
    detections = detect_debris(image)
    print(f"[DETECT] Found {len(detections)} debris items")

    ctx = SurveyContext(
        detections=detections,
        gps_lat=gps_lat,
        gps_lon=gps_lon,
    )

    print("[PHASE 1] Tool dispatch ...")
    tool_results = _run_phase1(ctx, language=language)

    print("[BUNDLE] Rendering data bundle ...")
    bundle = _build_data_bundle(tool_results, ctx)
    print(f"[BUNDLE] {len(bundle)} chars")

    print("[PHASE 2] Generating narrative on fresh conversation ...")
    report = _run_phase2(bundle, language=language)

    return report


print("Two-phase orchestrator defined.")

In [ ]:
"""End-to-end demo: load sample image -> detect -> tools -> report."""

import os
from pathlib import Path

# Sample image resolution order:
#   1. Kaggle dataset input:  /kaggle/input/oceanguard-samples/sample_debris.jpg
#   2. Manual upload to Kaggle notebook working dir: /kaggle/working/sample_debris.jpg
#   3. Fallback: synthetic 640x640 RGB image (dark ocean blue)
SAMPLE_PATHS = [
    Path("/kaggle/input/oceanguard-samples/sample_debris.jpg"),
    Path("/kaggle/working/sample_debris.jpg"),
]

sample_image: Image.Image | None = None
for candidate in SAMPLE_PATHS:
    if candidate.exists():
        sample_image = Image.open(candidate).convert("RGB")
        print(f"Loaded sample image from: {candidate}  size={sample_image.size}")
        break

if sample_image is None:
    print(
        "Sample image not found at any expected path. "
        "Using synthetic 640x640 placeholder. "
        "To use a real image, upload sample_debris.jpg to /kaggle/working/ "
        "or add it as a Kaggle dataset at /kaggle/input/oceanguard-samples/."
    )
    sample_image = Image.new("RGB", (640, 640), color=(10, 30, 60))

# GPS coordinates for demo (Mediterranean coastal site — fictional example)
DEMO_LAT = 38.7223
DEMO_LON = 0.1365

report_text = generate_grounded_report(
    image=sample_image,
    language="en",
    gps_lat=DEMO_LAT,
    gps_lon=DEMO_LON,
)

print("\n" + "=" * 70)
print("GENERATED REPORT")
print("=" * 70 + "\n")
print(report_text)

## Next: Validation, Multi-Language, Hallucination Repair (Cells 16-35 in v2)

The following cells are planned for the v2 release of this notebook:

| Cell | Topic | Android reference |
| --- | --- | --- |
| 16 | Full 7-tool OceanGuardTools port | `OceanGuardTools.kt` |
| 17 | `EnvironmentalImpact` table (degradation times, risk scores) | `EnvironmentalImpact.kt` |
| 18 | `ReportValidator` — 12-check hallucination detector | `ReportValidator.kt` |
| 19 | `repairHallucinations` — canonical row repair | `ToolReportGenerator.kt:121` |
| 20 | Multi-language demo (ES, FR, DE, IT, PT) | `ToolReportGenerator.kt:47-54` |
| 21 | Zone report variant (temporal trend tool) | `OceanGuardTools.getTemporalTrend` |
| 22 | Batch inference over multiple images | `DetectionOrchestrator.kt` |
| 23 | PDF export of validated report | `PdfReportExporter.kt` |
| 24-29 | Ablation study: baseline vs. two-phase vs. repaired | evaluation metrics |
| 30-32 | Real RT-DETRv2 ONNX inference (optional, requires model file) | `conversion/` |
| 33 | Visualize bounding boxes on sample image | `ui/components/BBoxOverlay.kt` |
| 34 | Token/latency benchmark vs. Android on-device numbers | `benchmark_exynos2200.md` |
| 35 | Summary table + competition submission metadata | — |

## 6. Complete Tool Suite — porting `OceanGuardTools.kt`

The Android app wires Gemma 4 to **seven no-argument tools** (see `OceanGuardTools.kt`).
Cells 1-15 implemented four of them (`get_total_debris_count`, `get_debris_by_category`,
`get_session_metadata`, `get_locations`). This cell adds the remaining three and extends
`execute_tool` to dispatch them, giving the full set used in production:

| Android method (camelCase) | Python snake_case | Returns |
| --- | --- | --- |
| `getDebrisSummary` | `get_debris_summary` | aggregate counts, avg health, dominant material/type, risk breakdown |
| `getMaterialBreakdown` | `get_material_breakdown` | count + % per material |
| `getTypeBreakdown` | `get_type_breakdown` | count + % per debris type |
| `getRiskAssessment` | `get_risk_assessment` | ranked risk table with urgency labels |
| `getCollectionWaypoints` | `get_collection_waypoints` | top-10 GPS waypoints (priority HIGH to LOW) |
| `getEcologicalImpacts` | `get_ecological_impacts` | degradation time, primary risk, risk score, annual ocean volume |
| `getSurveyStatistics` | `get_survey_statistics` | min/max/avg/median/stdDev for health score and debris count |

Note: `getTemporalTrend` is zone-report only and not included here (no multi-session data
in the single-image demo).


In [ ]:
"""Complete tool suite: 7 no-argument tools mirroring OceanGuardTools.kt."""
import math


# ---------------------------------------------------------------------------
# EnvironmentalImpact table — ported from EnvironmentalImpact.kt
# Data sources: NOAA Marine Debris Program, UNEP reports.
# ---------------------------------------------------------------------------
IMPACT_MAP: dict[str, dict[str, Any]] = {
    "Bottle":         {"degradation_time": "450+ years",    "primary_risk": "Ingestion risk, microplastic fragmentation",            "risk_score": 8,  "annual_volume": "~1M tonnes/year"},
    "Can":            {"degradation_time": "200+ years",    "primary_risk": "Sharp edges, toxic chemical leaching",                  "risk_score": 6,  "annual_volume": "~500K tonnes/year"},
    "Fishing_Net":    {"degradation_time": "600+ years",    "primary_risk": "Ghost fishing, entanglement of marine life",            "risk_score": 10, "annual_volume": "~640K tonnes/year"},
    "Glove":          {"degradation_time": "100+ years",    "primary_risk": "Ingestion risk, microplastic release",                  "risk_score": 7,  "annual_volume": None},
    "Mask":           {"degradation_time": "450+ years",    "primary_risk": "Entanglement of small marine organisms",                "risk_score": 7,  "annual_volume": None},
    "Metal_Debris":   {"degradation_time": "200-500 years", "primary_risk": "Sharp edges, chemical leaching, habitat disruption",    "risk_score": 6,  "annual_volume": None},
    "Plastic_Debris": {"degradation_time": "450+ years",    "primary_risk": "Microplastic fragmentation, ingestion by marine fauna", "risk_score": 8,  "annual_volume": "~8M tonnes/year"},
    "Tire":           {"degradation_time": "2000+ years",   "primary_risk": "Toxic chemicals (zinc, cadmium, heavy metals)",         "risk_score": 9,  "annual_volume": None},
}
_FALLBACK_IMPACT: dict[str, Any] = {
    "degradation_time": "Variable",
    "primary_risk": "Unknown environmental impact",
    "risk_score": 3,
    "annual_volume": None,
}


def _get_impact(label: str) -> dict[str, Any]:
    """Return impact info for a debris label, falling back to generic entry."""
    return IMPACT_MAP.get(label, _FALLBACK_IMPACT)


def _urgency_label(risk_score: int) -> str:
    """Map risk score to urgency label (mirrors OceanGuardTools.urgencyLabel)."""
    if risk_score >= 8:
        return "CRITICAL"
    if risk_score >= 6:
        return "HIGH"
    if risk_score >= 4:
        return "MEDIUM"
    return "LOW"


def _summarize_values(values: list[float]) -> dict[str, Any]:
    """Compute descriptive stats (mirrors OceanGuardTools.summarize)."""
    n = len(values)
    if n == 0:
        return {"n": 0}
    sorted_v = sorted(values)
    avg = sum(values) / n
    variance = sum((v - avg) ** 2 for v in values) / n
    return {
        "n": n,
        "min": round(sorted_v[0], 1),
        "max": round(sorted_v[-1], 1),
        "avg": round(avg, 1),
        "median": round(sorted_v[n // 2], 1),
        "std_dev": round(math.sqrt(variance), 1),
    }


# Extended tool schemas (3 new tools appended to TOOL_SCHEMAS from Cell 10)
EXTENDED_TOOL_SCHEMAS: list[dict[str, Any]] = TOOL_SCHEMAS + [
    {
        "type": "function",
        "function": {
            "name": "get_material_breakdown",
            "description": (
                "Get counts and percentages for each debris MATERIAL detected. "
                "Returned items list is authoritative — do not add other materials. "
                "Percentages sum to 100.0."
            ),
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_risk_assessment",
            "description": (
                "Get ranked risk table for each detected debris type with risk score (1-10), "
                "primary ecological risk, and urgency label (CRITICAL/HIGH/MEDIUM/LOW)."
            ),
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_ecological_impacts",
            "description": (
                "Get environmental persistence (degradation time), primary risk, risk score, "
                "and annual ocean volume for every detected debris type. "
                "Call this BEFORE writing about ecological impact."
            ),
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_survey_statistics",
            "description": (
                "Compute min/max/avg/median/stdDev for both health_score and debris_count "
                "across all analyzed images in a single call."
            ),
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
]

EXTENDED_REQUIRED_TOOLS: set[str] = {s["function"]["name"] for s in EXTENDED_TOOL_SCHEMAS}


def execute_tool_extended(name: str, ctx: SurveyContext) -> Any:
    """Dispatch all 8 tools; delegates to original execute_tool for the first 4."""
    original_tools = {
        "get_total_debris_count", "get_debris_by_category",
        "get_session_metadata", "get_locations",
    }
    if name in original_tools:
        return execute_tool(name, ctx)

    total = max(len(ctx.detections), 1)

    if name == "get_material_breakdown":
        items = [
            {"name": mat, "count": cnt, "percent": round(cnt * 100.0 / total, 1)}
            for mat, cnt in ctx.material_counts.items()
        ]
        return {"total": total, "items": items}

    if name == "get_risk_assessment":
        items = sorted(
            [
                {
                    "type": label,
                    "count": cnt,
                    "risk_score": _get_impact(label)["risk_score"],
                    "primary_risk": _get_impact(label)["primary_risk"],
                    "urgency": _urgency_label(_get_impact(label)["risk_score"]),
                }
                for label, cnt in ctx.type_counts.items()
            ],
            key=lambda x: x["risk_score"],  # type: ignore[index]
            reverse=True,
        )
        return {"items": items}

    if name == "get_ecological_impacts":
        items = [
            {
                "type": label,
                "count": cnt,
                "degradation_time": _get_impact(label)["degradation_time"],
                "primary_risk": _get_impact(label)["primary_risk"],
                "risk_score": _get_impact(label)["risk_score"],
                "annual_volume": _get_impact(label)["annual_volume"] or "Not quantified",
            }
            for label, cnt in ctx.type_counts.items()
        ]
        return {"count": len(items), "items": items}

    if name == "get_survey_statistics":
        health_values = [float(ctx.health_score)]
        count_values = [float(len(ctx.detections))]
        return {
            "health_score": _summarize_values(health_values),
            "debris_count": _summarize_values(count_values),
        }

    raise ValueError(f"Unknown tool: {name}")


# Smoke-test all 8 tools against the demo context
_smoke_ctx = SurveyContext(
    detections=_detections,
    gps_lat=DEMO_LAT,
    gps_lon=DEMO_LON,
)
for _tool_name in sorted(EXTENDED_REQUIRED_TOOLS):
    _result = execute_tool_extended(_tool_name, _smoke_ctx)
    print(f"  {_tool_name:35s} -> {json.dumps(_result)[:90]}")


## 7. Hallucination Repair — section-aware regex patterns

The Android `ToolReportGenerator.repairHallucinations()` (companion object, line 121)
repairs VLM output after PHASE 2 by matching each markdown table row label against the
canonical pre-rendered bundle. Section-aware matching avoids material/type collisions
when the same translated label appears in both tables with different numeric values.

Four helpers are ported directly from the companion object:

| Kotlin | Python | Purpose |
| --- | --- | --- |
| `normalizeLabel(s)` | `normalize_label(s)` | NFD diacritics strip, lowercase, collapse whitespace |
| `buildNormalizedMap(rows, name, other?)` | `build_normalized_map(rows, map_name, other_rows?)` | normalized-key lookup |
| `repairHallucinations(text, canon)` | `repair_hallucinations(text, material_rows, type_rows, ...)` | row-by-row canonical replacement |
| `buildCanonDateRows(canon)` | `build_canon_date_rows(sessions)` | extract canonical date rows for temporal repair |

**Regex fidelity note:** Kotlin uses `\\p{InCombiningDiacriticalMarks}` Unicode category;
Python uses `unicodedata.category(c) == "Mn"` which is equivalent (`Mn` = Non-spacing Mark).
The `\\d{5,}` date-corruption detector is identical in both implementations.


In [ ]:
"""Hallucination repair helpers ported from ToolReportGenerator companion object."""


def normalize_label(s: str) -> str:
    """Lowercase, NFD-decompose, strip combining marks, collapse whitespace.

    Mirrors ToolReportGenerator.normalizeLabel (Kotlin companion object, line 79).
    Kotlin: Normalizer.Form.NFD + replace InCombiningDiacriticalMarks + \s+
    Python: unicodedata.normalize("NFD") + category "Mn" strip + re.sub \s+
    Both produce identical output for all 6 supported languages.
    """
    decomposed = unicodedata.normalize("NFD", s.strip().lower())
    stripped = "".join(c for c in decomposed if unicodedata.category(c) != "Mn")
    return re.sub(r"\s+", " ", stripped)


def build_normalized_map(
    rows: dict[str, str],
    map_name: str,
    other_rows: dict[str, str] | None = None,
) -> dict[str, str]:
    """Build normalized-key -> canonical-line lookup.

    Mirrors ToolReportGenerator.buildNormalizedMap (line 91).
    Warns on ambiguous keys shared between material and type maps.
    """
    result: dict[str, str] = {}
    for label, line in rows.items():
        key = normalize_label(label)
        result[key] = line
        if other_rows is not None:
            other_line = next(
                (v for k, v in other_rows.items() if normalize_label(k) == key), None
            )
            if other_line is not None and other_line != line:
                print(
                    f"  [WARN] Ambiguous canon label '{label}' (norm='{key}') "
                    f"differs between {map_name} and other map — "
                    "section-aware repair will resolve"
                )
    return result


def repair_hallucinations(
    text: str,
    material_rows: dict[str, str],
    type_rows: dict[str, str],
    eco_rows: dict[str, str] | None = None,
    risk_rows: dict[str, str] | None = None,
) -> str:
    """Replace hallucinated table rows with canonical ground-truth lines.

    Mirrors ToolReportGenerator.repairHallucinations (line 121).
    Section-aware: tracks current ### heading to choose material vs type map.
    Date repair: rows with 5+ consecutive digits are left untouched.
    """
    norm_material = build_normalized_map(material_rows, "materialRows", type_rows)
    norm_type = build_normalized_map(type_rows, "typeRows")
    norm_eco = build_normalized_map(eco_rows or {}, "ecoRows")
    norm_risk = build_normalized_map(risk_rows or {}, "riskRows")

    # Section-heading keyword -> normalized map (mirrors Kotlin sectionHeadings)
    SECTION_MAP = [
        ("material",   norm_material),
        ("type",       norm_type),
        ("ecological", norm_eco),
        ("eco",        norm_eco),
        ("risk",       norm_risk),
    ]

    lines = text.splitlines()
    replaced = 0
    current_norm_map: dict[str, str] = {}

    for i, line in enumerate(lines):
        stripped = line.strip()

        if stripped.startswith("###"):
            lower = stripped.lower()
            current_norm_map = next(
                (nm for kw, nm in SECTION_MAP if kw in lower), {}
            )
            continue

        if not stripped.startswith("|"):
            continue
        if "---" in stripped:
            continue

        parts = stripped.split("|")
        first_cell = parts[1].strip() if len(parts) >= 3 else ""
        if not first_cell:
            continue

        # Date corruption check: 5+ consecutive digits -> leave untouched
        if re.search(r"\d{5,}", line):
            continue

        norm_key = normalize_label(first_cell)
        canonical = (
            current_norm_map.get(norm_key)
            or norm_eco.get(norm_key)
            or norm_risk.get(norm_key)
        )
        if canonical is not None and canonical != stripped:
            lines[i] = canonical
            replaced += 1

    if replaced > 0:
        print(f"  [repair] Rewrote {replaced} row(s)")

    result = "\n".join(lines)
    # Strip blank/dot-only table rows (mirrors Kotlin trailing-whitespace removal)
    result = re.sub(r"^\s*\|[\s.|]+\|\s*$", "", result, flags=re.MULTILINE)
    result = re.sub(r"\n{3,}", "\n\n", result)
    return result


def build_canon_date_rows(sessions: list[dict[str, Any]]) -> dict[str, str]:
    """Extract canonical date -> row mapping for temporal tables.

    Mirrors ToolReportGenerator.buildCanonDateRows (line 189).
    Returns empty map: Canon does not yet expose date rows so corrupt dates
    fall through to the leave-untouched path.
    TODO: populate once zone-report temporal tables are implemented (cells 30+).
    """
    return {}


# Smoke-test with synthetic hallucinated output
_canon_material = {
    "Plastic": "| Plastic | 2 | 66.7% |",
    "Metal":   "| Metal   | 1 | 33.3% |",
}
_hallucinated_table = (
    "### Materials\n"
    "| Header | Count | % |\n"
    "| --- | --- | --- |\n"
    "| plastik | 5 | 99% |\n"
    "| Metal   | 1 | 33.3% |\n"
)
_repaired = repair_hallucinations(_hallucinated_table, _canon_material, {})
print("Repaired output:")
print(_repaired)


## 8. Validation — porting `ReportValidator.kt`

The Android `ReportValidator` object (913 lines) runs 12 independent checks against
source session data after PHASE 2. This cell ports all 12 checks as pure Python
functions with no Android dependencies.

| Check | Field key | What it detects |
| --- | --- | --- |
| 1 | `debris_classes` | Fabricated debris types not in source data |
| 2 | `materials` | Fabricated materials not in source data |
| 3 | `percentages` | Percentage claims that deviate >1.5pp from ground truth |
| 4 | `percentage_sum` | Table percentage columns not summing to ~100% |
| 5 | `internal_consistency` | Same entity assigned conflicting percentages in different sections |
| 6 | `data_coverage` | Significant items (>=10%) omitted from report |
| 7 | `dates` | Dates outside 1-day-padded survey date range |
| 8 | `gps_coordinates` | GPS values outside source bounding box (or fabricated when no GPS in source) |
| 9 | `health_characterization` | Tone mismatch (alarming for healthy sites, positive for critical) |
| 10 | `text_repetition` | Circular/copy-pasted sentences (>=15 words) |
| 11 | `placeholders` | Unfilled `[Insert...]`, `[TODO]`, `| ... |` tokens |
| 12 | `risk_score_range` | Risk scores outside valid 1-10 range |

Score formula (mirrors `computeOverallScore`): mean(PASS=100, WARNING=60, FAIL=0) across all checks.


In [ ]:
"""ReportValidator port — 12 hallucination and consistency checks."""
from __future__ import annotations
from dataclasses import dataclass, field as dc_field
from enum import Enum


class CheckStatus(Enum):
    PASS = "PASS"
    WARNING = "WARNING"
    FAIL = "FAIL"


@dataclass
class ValidationCheck:
    """Single validation check result."""
    field: str
    status: CheckStatus
    detail: str


@dataclass
class ValidationResult:
    """Aggregated result of all validation checks."""
    score: int
    passed: list[str] = dc_field(default_factory=list)
    failed: list[str] = dc_field(default_factory=list)
    warnings: list[str] = dc_field(default_factory=list)
    checks: list[ValidationCheck] = dc_field(default_factory=list)

    def summary(self) -> str:
        """One-line summary string."""
        return (
            f"score={self.score}/100  "
            f"({len(self.passed)}P/{len(self.warnings)}W/{len(self.failed)}F)"
        )


# Regex patterns — identical constants to ReportValidator.kt
_PCT_RE = re.compile(r"(\d{1,3}(?:\.\d{1,2})?)\s*%")
_GPS_RE = re.compile(r"-?\d{1,3}\.\d{3,6}")
_DATE_RE = re.compile(r"\d{4}-\d{2}-\d{2}")
_HEALTH_SCORE_RE = re.compile(r"health\s*score[:\s]*(\d{1,3})", re.IGNORECASE)
_RISK_SCORE_RE = re.compile(r"risk\s*score[:\s]*(\d+)", re.IGNORECASE)
_PCT_TOLERANCE = 1.5  # matches PERCENTAGE_TOLERANCE in ReportValidator.kt

ALL_DEBRIS_TYPES: set[str] = set(DEBRIS_CLASSES)
ALL_MATERIALS: set[str] = {"PLASTIC", "METAL", "NYLON", "RUBBER"}

_NEGATIVE_TONE = [
    "critical", "severe", "alarming", "heavily polluted", "heavily contaminated",
    "extremely degraded", "urgent crisis", "devastating",
]
_POSITIVE_TONE = [
    "pristine", "excellent condition", "minimal pollution", "very healthy",
    "negligible contamination", "virtually clean",
]
_PLACEHOLDER_TOKENS = [
    "[Insert", "[insert", "[TODO", "[todo", "[PLACEHOLDER",
    "[Value", "[value", "[Name", "[name", "| ... |", "|...|",
]


def _find_best_entity_match(
    line_lower: str,
    entity_aliases: dict[str, set[str]],
) -> str | None:
    """Longest-match entity finder (mirrors ReportValidator.findBestEntityMatch)."""
    best_key: str | None = None
    best_len = 0
    for key, aliases in entity_aliases.items():
        for alias in aliases:
            if len(alias) > best_len and alias in line_lower:
                best_key = key
                best_len = len(alias)
    return best_key


def _compute_score(checks: list[ValidationCheck]) -> int:
    """Mean score: PASS=100, WARNING=60, FAIL=0."""
    if not checks:
        return 100
    weights = {CheckStatus.PASS: 100, CheckStatus.WARNING: 60, CheckStatus.FAIL: 0}
    return round(sum(weights[c.status] for c in checks) / len(checks))


def validate_report(
    report_md: str,
    ground_truth: dict[str, Any],
    language: str = "en",
) -> ValidationResult:
    """Run all 12 checks against ground_truth and return a ValidationResult.

    ground_truth expected keys:
      detections: list[Detection]
      timestamp: str (YYYY-MM-DD)
      gps_lat: float | None
      gps_lon: float | None
      health_score: int
    """
    checks: list[ValidationCheck] = []
    text_lower = report_md.lower()
    detections: list[Detection] = ground_truth.get("detections", [])
    present_types: set[str] = {d.label for d in detections}
    present_materials: set[str] = {d.material for d in detections}
    total = max(len(detections), 1)

    type_counts_gt: dict[str, int] = {}
    for d in detections:
        type_counts_gt[d.label] = type_counts_gt.get(d.label, 0) + 1
    mat_counts_gt: dict[str, int] = {}
    for d in detections:
        mat_counts_gt[d.material] = mat_counts_gt.get(d.material, 0) + 1

    type_pcts = {lbl: cnt * 100.0 / total for lbl, cnt in type_counts_gt.items()}
    mat_pcts = {mat: cnt * 100.0 / total for mat, cnt in mat_counts_gt.items()}
    all_pcts = {**type_pcts, **mat_pcts}

    entity_aliases: dict[str, set[str]] = {
        lbl: {lbl.lower(), lbl.lower().replace("_", " ")} for lbl in present_types
    }
    entity_aliases.update({mat: {mat.lower()} for mat in present_materials})

    # --- Check 1: debris classes ---
    fabricated_types = [
        t for t in ALL_DEBRIS_TYPES
        if t.lower().replace("_", " ") in text_lower and t not in present_types
    ]
    checks.append(ValidationCheck(
        field="debris_classes",
        status=CheckStatus.FAIL if fabricated_types else CheckStatus.PASS,
        detail=(f"Fabricated types: {fabricated_types}" if fabricated_types
                else "All mentioned debris types present in source data"),
    ))

    # --- Check 2: materials ---
    fabricated_mats = [
        m for m in ALL_MATERIALS
        if m.lower() in text_lower and m not in present_materials
    ]
    checks.append(ValidationCheck(
        field="materials",
        status=CheckStatus.FAIL if fabricated_mats else CheckStatus.PASS,
        detail=(f"Fabricated materials: {fabricated_mats}" if fabricated_mats
                else "All mentioned materials present in source data"),
    ))

    # --- Check 3: percentage accuracy ---
    matched_correct = matched_wrong = 0
    wrong_details: list[str] = []
    for line in report_md.splitlines():
        line_lower = line.lower()
        for m in _PCT_RE.finditer(line):
            claimed = float(m.group(1))
            if claimed in (0.0, 100.0):
                continue
            key = _find_best_entity_match(line_lower, entity_aliases)
            if key and key in all_pcts:
                if abs(claimed - all_pcts[key]) <= _PCT_TOLERANCE:
                    matched_correct += 1
                else:
                    matched_wrong += 1
                    wrong_details.append(
                        f"{key}: claimed {claimed}%, actual {all_pcts[key]:.1f}%"
                    )
    total_matched = matched_correct + matched_wrong
    if total_matched == 0:
        pct_status, pct_detail = CheckStatus.WARNING, "No verifiable percentages found in report"
    elif matched_wrong == 0:
        pct_status, pct_detail = CheckStatus.PASS, f"All {matched_correct} verified percentages accurate"
    else:
        pct_status = CheckStatus.FAIL if matched_wrong > matched_correct else CheckStatus.WARNING
        pct_detail = f"{matched_wrong}/{total_matched} inaccurate: {'; '.join(wrong_details)}"
    checks.append(ValidationCheck(field="percentages", status=pct_status, detail=pct_detail))

    # --- Check 4: percentage sum ---
    table_sums: list[float] = []
    in_table = False
    pct_col_idx = -1
    table_pcts_acc: list[float] = []
    for line in report_md.splitlines():
        t = line.strip()
        if t.startswith("|") and t.endswith("|"):
            cells = [c.strip() for c in t.split("|") if c.strip()]
            if not in_table:
                pct_col_idx = next((i for i, c in enumerate(cells) if "%" in c), -1)
                if pct_col_idx >= 0:
                    in_table = True
                    table_pcts_acc = []
            elif "---" in t:
                pass
            elif pct_col_idx < len(cells):
                raw = cells[pct_col_idx].replace("%", "").strip()
                try:
                    table_pcts_acc.append(float(raw))
                except ValueError:
                    pass
        elif in_table and table_pcts_acc:
            table_sums.append(sum(table_pcts_acc))
            table_pcts_acc = []
            in_table = False
            pct_col_idx = -1
    if table_pcts_acc:
        table_sums.append(sum(table_pcts_acc))
    bad_sums = [s for s in table_sums if abs(s - 100.0) > 5.0]
    if table_sums:
        checks.append(ValidationCheck(
            field="percentage_sum",
            status=CheckStatus.PASS if not bad_sums else CheckStatus.FAIL,
            detail=("Table percentage columns sum correctly (~100%)" if not bad_sums
                    else f"Table percentages sum to {[f'{s:.1f}%' for s in bad_sums]} instead of ~100%"),
        ))

    # --- Check 5: internal consistency ---
    entity_pcts_map: dict[str, list[float]] = {}
    for line in report_md.splitlines():
        line_lower = line.lower()
        for m in _PCT_RE.finditer(line):
            v = float(m.group(1))
            if v in (0.0, 100.0):
                continue
            key = _find_best_entity_match(line_lower, entity_aliases)
            if key:
                entity_pcts_map.setdefault(key, []).append(v)
    inconsistent = [
        f"{k}: {'/'.join(f'{p}%' for p in sorted(set(vs)))}"
        for k, vs in entity_pcts_map.items()
        if max(set(vs)) - min(set(vs)) > 2.0
    ]
    checks.append(ValidationCheck(
        field="internal_consistency",
        status=CheckStatus.FAIL if inconsistent else CheckStatus.PASS,
        detail=(f"Contradictory percentages: {'; '.join(inconsistent)}" if inconsistent
                else "Percentages consistent across sections"),
    ))

    # --- Check 6: data coverage ---
    missing = [
        f"{lbl} ({type_pcts[lbl]:.0f}%)"
        for lbl in present_types
        if type_pcts[lbl] >= 10.0 and lbl.lower().replace("_", " ") not in text_lower
    ]
    checks.append(ValidationCheck(
        field="data_coverage",
        status=CheckStatus.WARNING if missing else CheckStatus.PASS,
        detail=(f"Significant items omitted: {', '.join(missing)}" if missing
                else "All significant types (>=10%) mentioned"),
    ))

    # --- Check 7: dates ---
    survey_date = ground_truth.get("timestamp", "")
    claimed_dates = _DATE_RE.findall(report_md)
    if not claimed_dates:
        checks.append(ValidationCheck(
            field="dates", status=CheckStatus.PASS,
            detail="No explicit dates to validate",
        ))
    else:
        bad_dates = [d for d in claimed_dates if d != survey_date]
        checks.append(ValidationCheck(
            field="dates",
            status=CheckStatus.FAIL if bad_dates else CheckStatus.PASS,
            detail=(f"{len(bad_dates)}/{len(claimed_dates)} dates outside survey range ({survey_date})"
                    if bad_dates else f"All {len(claimed_dates)} dates match survey date"),
        ))

    # --- Check 8: GPS coordinates ---
    gps_lat = ground_truth.get("gps_lat")
    gps_lon = ground_truth.get("gps_lon")
    gps_matches = [
        float(m) for m in _GPS_RE.findall(report_md)
        if len(m.split(".")[-1]) >= 3
    ]
    if gps_lat is None:
        if len(gps_matches) >= 2:
            checks.append(ValidationCheck(
                field="gps_coordinates", status=CheckStatus.WARNING,
                detail=f"Report contains {len(gps_matches)} GPS-like values but source has no GPS",
            ))
    else:
        margin = 0.001
        out_of_bounds = [
            v for v in gps_matches
            if not (gps_lat - margin <= v <= gps_lat + margin)
            and not (gps_lon - margin <= v <= gps_lon + margin)
        ]
        checks.append(ValidationCheck(
            field="gps_coordinates",
            status=CheckStatus.WARNING if out_of_bounds else CheckStatus.PASS,
            detail=(f"{len(out_of_bounds)} coordinates outside source bounding box"
                    if out_of_bounds else "GPS coordinates consistent with source data"),
        ))

    # --- Check 9: health characterization ---
    avg_health = ground_truth.get("health_score", 50)
    has_neg = any(t in text_lower for t in _NEGATIVE_TONE)
    has_pos = any(t in text_lower for t in _POSITIVE_TONE)
    if avg_health > 75 and has_neg and not has_pos:
        hc_status = CheckStatus.FAIL
        hc_detail = f"Alarming language but health score={avg_health} (good condition)"
    elif avg_health < 30 and has_pos and not has_neg:
        hc_status = CheckStatus.FAIL
        hc_detail = f"Positive language but health score={avg_health} (critical condition)"
    else:
        hc_status = CheckStatus.PASS
        hc_detail = f"Health characterization consistent with score={avg_health}"
    checks.append(ValidationCheck(
        field="health_characterization", status=hc_status, detail=hc_detail,
    ))

    # --- Check 10: text repetition ---
    clean = re.sub(r"\|[^\n]*\|", "", report_md)
    sentences = [
        re.sub(r"\s+", " ", s.strip().lower())
        for s in re.split(r"[.\n]+", clean)
        if len(s.split()) >= 15
    ]
    seen_s: dict[str, int] = {}
    for s in sentences:
        seen_s[s] = seen_s.get(s, 0) + 1
    dupes = sum(1 for v in seen_s.values() if v > 1)
    if dupes == 0:
        rep_status, rep_detail = CheckStatus.PASS, "No significant text repetition"
    elif dupes <= 3:
        rep_status, rep_detail = CheckStatus.WARNING, f"{dupes} repeated sentence(s)"
    else:
        rep_status, rep_detail = CheckStatus.FAIL, f"{dupes} repeated sentences (circular output)"
    checks.append(ValidationCheck(field="text_repetition", status=rep_status, detail=rep_detail))

    # --- Check 11: placeholders ---
    found_ph = [p for p in _PLACEHOLDER_TOKENS if p in report_md]
    checks.append(ValidationCheck(
        field="placeholders",
        status=CheckStatus.FAIL if found_ph else CheckStatus.PASS,
        detail=(f"Unfilled placeholders: {found_ph}" if found_ph
                else "No unfilled placeholders"),
    ))

    # --- Check 12: risk score range (1-10, matching IMPACT_MAP scale) ---
    risk_vals = [int(m) for m in _RISK_SCORE_RE.findall(report_md)]
    if risk_vals:
        out_range = [v for v in risk_vals if not (1 <= v <= 10)]
        checks.append(ValidationCheck(
            field="risk_score_range",
            status=CheckStatus.FAIL if out_range else CheckStatus.PASS,
            detail=(f"{len(out_range)}/{len(risk_vals)} risk scores outside 1-10 range: {out_range}"
                    if out_range else f"All {len(risk_vals)} risk scores within 1-10 range"),
        ))

    return ValidationResult(
        score=_compute_score(checks),
        passed=[c.field for c in checks if c.status == CheckStatus.PASS],
        failed=[c.field for c in checks if c.status == CheckStatus.FAIL],
        warnings=[c.field for c in checks if c.status == CheckStatus.WARNING],
        checks=checks,
    )


# Smoke-test with a known synthetic report
_test_report = (
    f"Survey date: {_smoke_ctx.timestamp}\n"
    "Health score: 25/100\n"
    "Plastic 66.7%\n"
    "Fishing_Net 27.0%\n"
)
_gt = {
    "detections": _smoke_ctx.detections,
    "timestamp": _smoke_ctx.timestamp,
    "gps_lat": DEMO_LAT,
    "gps_lon": DEMO_LON,
    "health_score": _smoke_ctx.health_score,
}
_vr = validate_report(_test_report, _gt)
print(_vr.summary())
for _chk in _vr.checks:
    print(f"  [{_chk.status.value}] {_chk.field}: {_chk.detail}")


## 9. Multilingual Demo — 6 Languages

OceanGuard AI generates reports in six languages (mirrors
`ToolReportGenerator.LANGUAGE_NAMES`, line 47):
`en`, `es`, `fr`, `de`, `it`, `pt`.

The loop below calls `generate_grounded_report_lang(image, language=lang)` for each
language and prints:
- first non-empty paragraph of the PHASE 2 output (sanity check)
- `ValidationResult.summary()` against the same ground-truth context

> **Runtime note**: each call runs the full two-phase pipeline (~90s on T4 per language).
> On CPU-only Kaggle sessions this cell may take ~10 minutes. Set
> `MULTILINGUAL_DEMO = False` to skip it during fast iteration.


In [ ]:
"""Multilingual demo: generate a grounded report in each of the 6 supported languages."""

# Set to False to skip during fast iteration
MULTILINGUAL_DEMO = True

_LANGUAGE_NAMES = {
    "en": "English",
    "es": "Spanish (Espanol)",
    "fr": "French (Francais)",
    "de": "German (Deutsch)",
    "it": "Italian (Italiano)",
    "pt": "Portuguese (Portugues)",
}

_LANG_WRITER_SYSTEM: dict[str, str] = {
    lang: (
        f"You are a senior marine conservation scientist. "
        f"Write the assessment in {name}. "
        "RULES: CLOSED-WORLD (only debris from CONFIRMED DATA), "
        "COPY TABLES VERBATIM, NO PLACEHOLDERS, readable labels (no UPPER_SNAKE_CASE). "
        f"OUTPUT LANGUAGE: {name}."
    )
    for lang, name in _LANGUAGE_NAMES.items()
}


def generate_grounded_report_lang(
    image: Image.Image,
    language: str = "en",
    gps_lat: float | None = None,
    gps_lon: float | None = None,
) -> tuple[str, ValidationResult]:
    """Two-phase pipeline with language-specific writer system and validation."""
    detections = detect_debris(image)
    ctx = SurveyContext(detections=detections, gps_lat=gps_lat, gps_lon=gps_lon)
    tool_results = _run_phase1(ctx, language=language)
    bundle = _build_data_bundle(tool_results, ctx)

    writer_sys = _LANG_WRITER_SYSTEM.get(language, _LANG_WRITER_SYSTEM["en"])
    messages = [
        {"role": "system", "content": writer_sys},
        {"role": "user",   "content": bundle},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS_P2,
            do_sample=True,
            temperature=0.3,
            top_k=20,
            pad_token_id=tokenizer.eos_token_id,
        )
    report = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True
    )

    gt: dict[str, Any] = {
        "detections": detections,
        "timestamp": ctx.timestamp,
        "gps_lat": gps_lat,
        "gps_lon": gps_lon,
        "health_score": ctx.health_score,
    }
    vr = validate_report(report, gt, language=language)
    return report, vr


if MULTILINGUAL_DEMO:
    print("Running multilingual demo (6 languages)...")
    print("=" * 70)
    for lang_code, lang_name in _LANGUAGE_NAMES.items():
        print(f"\n--- {lang_name} ({lang_code}) ---")
        try:
            report_lang, vr_lang = generate_grounded_report_lang(
                sample_image, language=lang_code,
                gps_lat=DEMO_LAT, gps_lon=DEMO_LON,
            )
            first_para = next(
                (p.strip() for p in report_lang.split("\n\n") if p.strip()), ""
            )
            print(f"First paragraph: {first_para[:200]}")
            print(f"Validation: {vr_lang.summary()}")
        except Exception as exc:
            print(f"  [ERROR] {exc}")
    print("=" * 70)
else:
    print("MULTILINGUAL_DEMO=False — skipped. Set True to run full 6-language demo.")


## 10. Ablation: Two-Phase vs Naive Prompt

This cell compares three conditions to quantify the benefit of the two-phase
tool-calling architecture:

| Condition | Description |
| --- | --- |
| **Naive** | Single prompt: detection labels injected as raw text, model generates prose in one shot with no tool calling |
| **Two-Phase** | PHASE 1 tool calling + PHASE 2 fresh conversation with pre-rendered bundle (production approach) |
| **Two-Phase + Repair** | Two-Phase output passed through `repair_hallucinations()` (canonical row substitution) |

Metrics per condition:
- `validation_score`: from `validate_report()` (0-100)
- `hallucination_checks_failed`: number of FAIL checks from the 12-check suite
- `latency_s`: wall-clock seconds from image input to final report string

> All three conditions use `do_sample=False` (temperature=0) for reproducibility
> across Kaggle sessions.


In [ ]:
"""Ablation study: Naive vs Two-Phase vs Two-Phase+Repair."""
import time


def _generate_naive(detections: list[Detection], ctx: SurveyContext) -> str:
    """Baseline: single-turn prompt, no tool calling, detections as plain text."""
    det_text = ", ".join(
        f"{d.label} (conf={d.confidence:.2f})" for d in detections
    )
    system_msg = (
        "You are a marine biologist. Write a short debris assessment report "
        "based on the detections provided. Use markdown."
    )
    user_msg = (
        f"Survey date: {ctx.timestamp}\n"
        f"GPS: {ctx.gps_lat}, {ctx.gps_lon}\n"
        f"Detections: {det_text}\n"
        "Write a 300-word assessment."
    )
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user",   "content": user_msg},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True
    )


def _run_ablation(image: Image.Image) -> list[dict[str, Any]]:
    """Run all 3 conditions and return comparison rows."""
    detections = detect_debris(image)
    ctx = SurveyContext(detections=detections, gps_lat=DEMO_LAT, gps_lon=DEMO_LON)
    gt: dict[str, Any] = {
        "detections": detections,
        "timestamp": ctx.timestamp,
        "gps_lat": DEMO_LAT,
        "gps_lon": DEMO_LON,
        "health_score": ctx.health_score,
    }
    results: list[dict[str, Any]] = []

    # Condition 1: Naive
    t0 = time.perf_counter()
    naive_report = _generate_naive(detections, ctx)
    naive_latency = round(time.perf_counter() - t0, 1)
    naive_vr = validate_report(naive_report, gt)
    results.append({
        "condition": "Naive",
        "validation_score": naive_vr.score,
        "hallucination_checks_failed": len(naive_vr.failed),
        "latency_s": naive_latency,
    })

    # Condition 2: Two-Phase (temperature=0 for ablation reproducibility)
    t0 = time.perf_counter()
    tool_results = _run_phase1(ctx)
    bundle = _build_data_bundle(tool_results, ctx)
    writer_sys = (
        "You are a senior marine conservation scientist. "
        "RULES: CLOSED-WORLD, COPY TABLES VERBATIM, NO PLACEHOLDERS. "
        "Write the report in English."
    )
    messages = [
        {"role": "system", "content": writer_sys},
        {"role": "user",   "content": bundle},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS_P2,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    two_phase_report = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True
    )
    two_phase_latency = round(time.perf_counter() - t0, 1)
    two_phase_vr = validate_report(two_phase_report, gt)
    results.append({
        "condition": "Two-Phase",
        "validation_score": two_phase_vr.score,
        "hallucination_checks_failed": len(two_phase_vr.failed),
        "latency_s": two_phase_latency,
    })

    # Condition 3: Two-Phase + Repair
    t0 = time.perf_counter()
    cat_items = tool_results.get("get_debris_by_category", {}).get("items", [])
    mat_items = execute_tool_extended("get_material_breakdown", ctx).get("items", [])
    canon_type_rows = {
        item["name"]: f"| {item['name']} | {item['count']} | {item['percent']}% |"
        for item in cat_items
    }
    canon_mat_rows = {
        item["name"]: f"| {item['name']} | {item['count']} | {item['percent']}% |"
        for item in mat_items
    }
    repaired_report = repair_hallucinations(two_phase_report, canon_mat_rows, canon_type_rows)
    repair_latency = round(time.perf_counter() - t0, 1)
    repaired_vr = validate_report(repaired_report, gt)
    results.append({
        "condition": "Two-Phase+Repair",
        "validation_score": repaired_vr.score,
        "hallucination_checks_failed": len(repaired_vr.failed),
        "latency_s": two_phase_latency + repair_latency,
    })

    return results


print("Running ablation study (3 conditions x 1 image)...")
ablation_rows = _run_ablation(sample_image)

print()
print(f"{'Condition':<25} {'Score':>7} {'Halluc. Fails':>14} {'Latency (s)':>12}")
print("-" * 62)
for row in ablation_rows:
    print(
        f"{row['condition']:<25} "
        f"{row['validation_score']:>7} "
        f"{row['hallucination_checks_failed']:>14} "
        f"{row['latency_s']:>12.1f}"
    )

if len(ablation_rows) >= 2:
    naive_score = ablation_rows[0]["validation_score"]
    best_score = max(r["validation_score"] for r in ablation_rows[1:])
    print(f"\nTwo-phase improvement over naive: +{best_score - naive_score} validation points")


## 11. Zone Reports — Multi-session Aggregation

The Android `ToolReportGenerator` produces two report variants:

| Variant | Input | Key extra tool |
| --- | --- | --- |
| **Generic** | 1 session | — |
| **Zone** | N sessions | `getTemporalTrend` |

A zone report aggregates multiple `DetectionSession` records (different timestamps,
same geographic zone) and exposes a temporal trend so Gemma 4 can describe whether
pollution is improving, worsening, or stable.

This cell:
1. Defines `SessionSnapshot` and `ZoneSurveyContext` extending `SurveyContext`.
2. Implements `get_temporal_trend` (mirrors `OceanGuardTools.getTemporalTrend`).
3. Builds a `ZoneSurveyContext` with 5 simulated sessions spanning 28 days.
4. Runs the two-phase pipeline over the aggregated context.


In [ ]:
"""Zone report demo: temporal trend tool + multi-session aggregation."""
from dataclasses import dataclass, field as dc_field2
from datetime import datetime, timedelta


@dataclass
class SessionSnapshot:
    """Minimal session record used for zone aggregation."""

    timestamp_ms: int
    detections: list[Detection]
    gps_lat: float | None = None
    gps_lon: float | None = None

    @property
    def health_score(self) -> int:
        """0-100 score: 100 - 25*count, floored at 0."""
        return max(0, 100 - 25 * len(self.detections))

    @property
    def total_count(self) -> int:
        return len(self.detections)


@dataclass
class ZoneSurveyContext(SurveyContext):
    """Extended SurveyContext for zone reports: holds multiple sessions."""

    sessions: list[SessionSnapshot] = dc_field2(default_factory=list)

    @property
    def temporal_trend(self) -> dict[str, Any] | None:
        """Day-by-day trend from sorted sessions; None when fewer than 2 sessions."""
        if len(self.sessions) < 2:
            return None
        sorted_s = sorted(self.sessions, key=lambda s: s.timestamp_ms)
        days = [
            {
                "date_ms": s.timestamp_ms,
                "session_count": 1,
                "total_debris": s.total_count,
                "avg_health_score": s.health_score,
            }
            for s in sorted_s
        ]
        delta = days[-1]["avg_health_score"] - days[0]["avg_health_score"]
        trend = "IMPROVING" if delta >= 5 else "WORSENING" if delta <= -5 else "STABLE"
        return {"trend": trend, "health_score_delta": delta, "days": days}


def get_temporal_trend_tool(ctx: ZoneSurveyContext) -> dict[str, Any]:
    """Tool: temporal evolution of debris counts and health scores across survey days."""
    trend = ctx.temporal_trend
    if trend is None:
        return {"available": False}
    return {"available": True, **trend}


# Build 5-session ZoneSurveyContext
ZONE_LAT, ZONE_LON = 41.3851, 2.1734  # Barcelona coast (fictional survey)
_BASE_DATE = datetime(2025, 3, 1)
_SESSION_COUNTS = [2, 2, 3, 3, 4]  # worsening trend over 28 days

zone_sessions: list[SessionSnapshot] = []
for _i, _count in enumerate(_SESSION_COUNTS):
    _ts = _BASE_DATE + timedelta(days=_i * 7)
    _ts_ms = int(_ts.timestamp() * 1000)
    random.seed(SEED + _i)
    _dets = [
        Detection(
            label=random.choice(DEBRIS_CLASSES),
            confidence=round(random.uniform(0.65, 0.95), 2),
            box=[
                round(random.uniform(0.05, 0.40), 2),
                round(random.uniform(0.05, 0.40), 2),
                round(random.uniform(0.55, 0.90), 2),
                round(random.uniform(0.55, 0.90), 2),
            ],
            material=_CLASS_TO_MATERIAL.get(random.choice(DEBRIS_CLASSES), "PLASTIC"),
        )
        for _ in range(_count)
    ]
    zone_sessions.append(
        SessionSnapshot(
            timestamp_ms=_ts_ms,
            detections=_dets,
            gps_lat=round(ZONE_LAT + random.uniform(-0.002, 0.002), 4),
            gps_lon=round(ZONE_LON + random.uniform(-0.002, 0.002), 4),
        )
    )

_all_zone_dets = [d for s in zone_sessions for d in s.detections]
zone_ctx = ZoneSurveyContext(
    detections=_all_zone_dets,
    gps_lat=ZONE_LAT,
    gps_lon=ZONE_LON,
    sessions=zone_sessions,
)

trend_data = get_temporal_trend_tool(zone_ctx)
print(f"Zone sessions: {len(zone_sessions)}")
print(f"Total debris across zone: {len(_all_zone_dets)}")
print(f"Trend: {trend_data['trend']}  health_delta={trend_data['health_score_delta']}")
print()

# Run two-phase pipeline over zone context
ZONE_REPORT_ENABLED = True  # set False to skip on CPU-only sessions

if ZONE_REPORT_ENABLED:
    print("[ZONE REPORT] Two-phase pipeline over 5-session zone context...")
    _zone_tool_results = _run_phase1(zone_ctx)
    _zone_tool_results["get_temporal_trend"] = get_temporal_trend_tool(zone_ctx)
    _zone_bundle = _build_data_bundle(_zone_tool_results, zone_ctx)
    zone_report = _run_phase2(_zone_bundle)
    print("[ZONE REPORT] First 400 chars:")
    print(zone_report[:400])
else:
    print("ZONE_REPORT_ENABLED=False — skipping zone report generation.")


## 12. Batch Inference — DetectionOrchestrator Pattern

The Android `DetectionOrchestrator` (`inference/DetectionOrchestrator.kt`) runs the
detection pipeline sequentially per image, emitting partial results via `StateFlow`
for progressive UI updates. Relevant batch behaviour:

- RT-DETRv2 runs first (fast, ~6s on Exynos 2200) and emits `DetectionsReady`.
- VLM runs second (optional, only when debris is found).
- `AnalysisResult` combines both into a single summary per image.

This cell replicates the aggregation logic in Python:
1. Loops over 3 images (same placeholder, different random seeds for mock variety).
2. Runs `detect_debris` + (optionally) `generate_grounded_report` per image.
3. Validates each report with `validate_report()`.
4. Aggregates `ValidationResult` objects and prints a cross-batch summary.

Set `BATCH_RUN_VLM = False` to run detection + validation only (fast path).


In [ ]:
"""Batch inference: 3 images, aggregate ValidationResult across batch."""
import time


BATCH_RUN_VLM = True   # set False to skip VLM and run detection+validation only
BATCH_SEEDS = [SEED, SEED + 10, SEED + 20]


def _make_batch_image(seed: int) -> Image.Image:
    """Synthetic 640x640 image with unique per-seed color tint."""
    random.seed(seed)
    r = random.randint(0, 20)
    g = random.randint(20, 50)
    b = random.randint(40, 80)
    return Image.new("RGB", (640, 640), color=(r, g, b))


def _detect_with_seed(seed: int) -> list[Detection]:
    """Deterministic mock detections with per-seed variation."""
    random.seed(seed)
    count = random.randint(2, 4)
    return [
        Detection(
            label=random.choice(DEBRIS_CLASSES),
            confidence=round(random.uniform(0.60, 0.95), 2),
            box=[
                round(random.uniform(0.05, 0.40), 2),
                round(random.uniform(0.05, 0.40), 2),
                round(random.uniform(0.60, 0.95), 2),
                round(random.uniform(0.60, 0.95), 2),
            ],
            material=_CLASS_TO_MATERIAL.get(random.choice(DEBRIS_CLASSES), "PLASTIC"),
        )
        for _ in range(count)
    ]


@dataclass
class BatchImageResult:
    """Per-image result in the batch pipeline."""

    image_idx: int
    detection_count: int
    health_score: int
    validation_score: int
    checks_failed: int
    checks_warned: int
    latency_s: float


batch_results: list[BatchImageResult] = []

for _bidx, _bseed in enumerate(BATCH_SEEDS):
    _bt0 = time.perf_counter()
    _bimg = _make_batch_image(_bseed)
    _bdets = _detect_with_seed(_bseed)
    _bctx = SurveyContext(
        detections=_bdets,
        gps_lat=DEMO_LAT + _bidx * 0.001,
        gps_lon=DEMO_LON + _bidx * 0.001,
    )
    _bgt: dict[str, Any] = {
        "detections": _bdets,
        "timestamp": _bctx.timestamp,
        "gps_lat": _bctx.gps_lat,
        "gps_lon": _bctx.gps_lon,
        "health_score": _bctx.health_score,
    }

    if BATCH_RUN_VLM:
        _breport = generate_grounded_report(
            _bimg, gps_lat=_bctx.gps_lat, gps_lon=_bctx.gps_lon
        )
    else:
        _btools = {t: execute_tool(t, _bctx) for t in REQUIRED_TOOLS}
        _breport = _build_data_bundle(_btools, _bctx)

    _bvr = validate_report(_breport, _bgt)
    _blat = round(time.perf_counter() - _bt0, 1)
    batch_results.append(
        BatchImageResult(
            image_idx=_bidx + 1,
            detection_count=len(_bdets),
            health_score=_bctx.health_score,
            validation_score=_bvr.score,
            checks_failed=len(_bvr.failed),
            checks_warned=len(_bvr.warnings),
            latency_s=_blat,
        )
    )
    print(
        f"Image {_bidx+1}: dets={len(_bdets)}  health={_bctx.health_score}  "
        f"val={_bvr.score}  fails={len(_bvr.failed)}  t={_blat}s"
    )

_mean_score = round(sum(r.validation_score for r in batch_results) / len(batch_results), 1)
_total_failed = sum(r.checks_failed for r in batch_results)
_total_warned = sum(r.checks_warned for r in batch_results)
_total_lat = round(sum(r.latency_s for r in batch_results), 1)

print()
print(f"{'Image':>7} {'Detections':>11} {'Health':>7} {'ValScore':>9} {'Fails':>6} {'Warns':>6} {'Latency':>9}")
print("-" * 62)
for _r in batch_results:
    print(
        f"{_r.image_idx:>7} {_r.detection_count:>11} {_r.health_score:>7} "
        f"{_r.validation_score:>9} {_r.checks_failed:>6} {_r.checks_warned:>6} "
        f"{_r.latency_s:>8.1f}s"
    )
print("-" * 62)
print(
    f"{'MEAN/TOTAL':>7} {'':>11} {'':>7} {_mean_score:>9} "
    f"{_total_failed:>6} {_total_warned:>6} {_total_lat:>8.1f}s"
)


## 13. Real RT-DETRv2 Inference (Optional)

The Android app runs RT-DETRv2 as a `.tflite` model with the **XNNPACK CPU delegate**
(8 threads, ~6s on Exynos 2200). The notebook provides an optional path using the
equivalent ONNX checkpoint via `onnxruntime`.

**Topology note**: the `.tflite` and `.onnx` files share the same RT-DETRv2 backbone
(ResNet-18d encoder, deformable attention decoder). The Android TFLite version uses a
`tanh`-approximation surgery to replace the Erf operator (unsupported on some delegates).
See `conversion/` scripts in the repo.

To activate this cell, attach the model as a Kaggle dataset at:
`/kaggle/input/oceanguard-models/rtdetrv2_detector.onnx`

If the file is absent the cell prints a skip message and the notebook continues
using the mock detector from cells 9-25.


In [ ]:
"""Real RT-DETRv2 ONNX inference — conditional on model availability."""
from pathlib import Path

ONNX_MODEL_PATH = Path("/kaggle/input/oceanguard-models/rtdetrv2_detector.onnx")
RTDETR_INPUT_SIZE = 640        # matches RTDETRInference.INPUT_SIZE in Android
RTDETR_CONF_THRESH = 0.5       # matches Android default confidenceThreshold


def _preprocess_rtdetr(image: Image.Image) -> "np.ndarray":
    """Resize to 640x640, normalize [0,1], return NCHW float32 array."""
    import numpy as np
    resized = image.resize((RTDETR_INPUT_SIZE, RTDETR_INPUT_SIZE), Image.BILINEAR)
    arr = np.array(resized, dtype=np.float32) / 255.0  # HWC
    arr = arr.transpose(2, 0, 1)                        # CHW
    return arr[np.newaxis]                               # NCHW (1,3,640,640)


def _postprocess_rtdetr(
    boxes: "np.ndarray",
    logits: "np.ndarray",
    threshold: float = RTDETR_CONF_THRESH,
) -> list[Detection]:
    """Sigmoid + threshold on RT-DETRv2 output (no extra NMS: model is query-based).

    Expected shapes (match Android RTDETRInference.kt):
        boxes:  (1, 150, 4)  normalized cxcywh
        logits: (1, 150, 8)  raw class logits
    """
    import numpy as np

    scores = 1.0 / (1.0 + np.exp(-logits[0]))   # sigmoid -> (150, 8)
    class_ids = scores.argmax(axis=-1)            # (150,)
    confidences = scores.max(axis=-1)             # (150,)
    keep = confidences >= threshold

    results: list[Detection] = []
    for i in np.where(keep)[0]:
        cx, cy, w, h = boxes[0, i]
        label = DEBRIS_CLASSES[int(class_ids[i]) % len(DEBRIS_CLASSES)]
        results.append(Detection(
            label=label,
            confidence=float(confidences[i]),
            box=[float(cx - w / 2), float(cy - h / 2),
                 float(cx + w / 2), float(cy + h / 2)],
            material=_CLASS_TO_MATERIAL.get(label, "PLASTIC"),
        ))
    return results


if ONNX_MODEL_PATH.exists():
    import numpy as np
    import onnxruntime as ort
    import time as _rt

    _sess_opts = ort.SessionOptions()
    _sess_opts.intra_op_num_threads = 4
    _ort_sess = ort.InferenceSession(
        str(ONNX_MODEL_PATH),
        sess_options=_sess_opts,
        providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
    )
    _in_name = _ort_sess.get_inputs()[0].name
    print(f"ONNX model loaded: {ONNX_MODEL_PATH.name}")
    print(f"Input: {_in_name}  {_ort_sess.get_inputs()[0].shape}")
    print(f"Outputs: {[o.name for o in _ort_sess.get_outputs()]}")

    _inp = _preprocess_rtdetr(sample_image)
    _t0 = _rt.perf_counter()
    _ort_out = _ort_sess.run(None, {_in_name: _inp})
    _ort_ms = round((_rt.perf_counter() - _t0) * 1000)

    _onnx_dets = _postprocess_rtdetr(_ort_out[0], _ort_out[1])
    print(f"Inference: {_ort_ms} ms  (ONNX Runtime, Kaggle T4)")
    print(f"Detections above threshold {RTDETR_CONF_THRESH}: {len(_onnx_dets)}")
    for _d in _onnx_dets:
        print(f"  {_d.label:20s}  conf={_d.confidence:.2f}  box={[round(v,3) for v in _d.box]}")
    print()
    print("NOTE: Android uses the same topology as a TFLite FP16/INT8 model with XNNPACK "
          "CPU delegate (8 threads). The TFLite variant uses a tanh approximation in "
          "place of Erf (see conversion/erf_surgery.py).")
else:
    print(
        "Skipping — ONNX model not present. "
        "Reproducible flow runs in mock mode (cells 9-25). "
        "To enable real inference, attach a Kaggle dataset containing "
        "rtdetrv2_detector.onnx at /kaggle/input/oceanguard-models/."
    )


## 14. Visualization — Bounding Box Rendering

The Android app renders bounding boxes via `BitmapAnnotator.kt` (static JPEG export)
and a Compose `BoundingBoxOverlay` component (live camera feed).

Both use **L-shaped corner brackets** with:
- Arm length: 10–15% of the shorter box side, clamped to [6, 80] px.
- Class-specific color (matching `BitmapAnnotator.CLASS_COLOR_MAP`).
- Semi-transparent fill (10% alpha of class color).
- Pill-shaped label chip above the top-left corner.

This cell replicates that style using `PIL.ImageDraw` and renders the three
detections from the main demo onto `sample_image`.


In [ ]:
"""Corner-bracket bounding box rendering — mirrors BitmapAnnotator.kt style."""
from PIL import Image as _PilImg, ImageDraw, ImageFont
from IPython.display import display

# Class color palette (RGB) mirrors BitmapAnnotator.CLASS_COLOR_MAP
_BBOX_COLORS: dict[str, tuple[int, int, int]] = {
    "Bottle":         (233, 30,  99),
    "Can":            (158, 158, 158),
    "Fishing_Net":    (255, 111, 0),
    "Glove":          (63,  81,  181),
    "Mask":           (233, 30,  99),
    "Metal_Debris":   (158, 158, 158),
    "Plastic_Debris": (233, 30,  99),
    "Tire":           (121, 85,  72),
}
_DEFAULT_BBOX_COLOR: tuple[int, int, int] = (96, 125, 139)


def _bbox_color(label: str) -> tuple[int, int, int]:
    return _BBOX_COLORS.get(label, _DEFAULT_BBOX_COLOR)


def _draw_corner_brackets(
    draw: ImageDraw.ImageDraw,
    x1: float,
    y1: float,
    x2: float,
    y2: float,
    color: tuple[int, int, int],
    stroke: int = 3,
) -> None:
    """Four L-shaped corners; arm = 10% of shorter side, clamped to [6, 80] px."""
    arm = int(max(6, min(80, 0.10 * min(x2 - x1, y2 - y1))))
    # top-left
    draw.line([(x1, y1), (x1 + arm, y1)], fill=color, width=stroke)
    draw.line([(x1, y1), (x1, y1 + arm)], fill=color, width=stroke)
    # top-right
    draw.line([(x2, y1), (x2 - arm, y1)], fill=color, width=stroke)
    draw.line([(x2, y1), (x2, y1 + arm)], fill=color, width=stroke)
    # bottom-left
    draw.line([(x1, y2), (x1 + arm, y2)], fill=color, width=stroke)
    draw.line([(x1, y2), (x1, y2 - arm)], fill=color, width=stroke)
    # bottom-right
    draw.line([(x2, y2), (x2 - arm, y2)], fill=color, width=stroke)
    draw.line([(x2, y2), (x2, y2 - arm)], fill=color, width=stroke)


def annotate_image(
    image: Image.Image,
    detections: list[Detection],
    stroke: int = 3,
) -> Image.Image:
    """Draw corner-bracket boxes and label chips onto a copy of the image."""
    annotated = image.copy().convert("RGBA")
    W, H = annotated.size

    # Separate overlay for semi-transparent fill (alpha composite later)
    fill_layer = _PilImg.new("RGBA", (W, H), (0, 0, 0, 0))
    fill_draw = ImageDraw.Draw(fill_layer)
    top_draw = ImageDraw.Draw(annotated)

    try:
        _font = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 16
        )
    except OSError:
        _font = ImageFont.load_default()

    for det in detections:
        x1 = int(det.box[0] * W)
        y1 = int(det.box[1] * H)
        x2 = int(det.box[2] * W)
        y2 = int(det.box[3] * H)
        color = _bbox_color(det.label)

        # 1. Semi-transparent fill (10% alpha = 26/255)
        fill_draw.rectangle([x1, y1, x2, y2], fill=color + (26,))

        # 2. Corner brackets
        _draw_corner_brackets(top_draw, x1, y1, x2, y2, color, stroke)

        # 3. Label chip
        label_text = f"{det.label} {det.confidence:.0%}"
        _bbox = top_draw.textbbox((0, 0), label_text, font=_font)
        _tw = _bbox[2] - _bbox[0]
        _th = _bbox[3] - _bbox[1]
        _pad = 4
        _chip_y0 = max(0, y1 - _th - _pad * 2 - stroke)
        _chip_y1 = _chip_y0 + _th + _pad * 2
        _chip_x1 = min(x1 + _tw + _pad * 2, W)
        top_draw.rounded_rectangle(
            [x1, _chip_y0, _chip_x1, _chip_y1],
            radius=4,
            fill=color + (217,),
        )
        top_draw.text((x1 + _pad, _chip_y0 + _pad), label_text, fill=(255, 255, 255), font=_font)

    result = _PilImg.alpha_composite(annotated, fill_layer).convert("RGB")
    return result


# Render demo detections onto sample_image
_viz_dets = detect_debris(sample_image)
annotated_image = annotate_image(sample_image, _viz_dets)

print(f"Image size: {annotated_image.size}  detections rendered: {len(_viz_dets)}")
for _vd in _viz_dets:
    print(f"  {_vd.label:20s}  conf={_vd.confidence:.2f}  box={[round(v, 3) for v in _vd.box]}")
display(annotated_image)


## 15. Performance Benchmarks — Exynos 2200 Reference

All figures were measured on a **Samsung Galaxy S22 Ultra** (Exynos 2200, CPU-only;
Xclipse 920 GPU not supported by either backend) on 2026-04-10.
Source: `memory/benchmark_exynos2200.md`.

The code cell below:
1. Prints the full benchmark table.
2. Computes expected total report latency for 700–900 word outputs
   (estimate: 1.3 tokens/word).
3. Closes the notebook with submission metadata.


In [ ]:
"""Performance benchmarks — Exynos 2200 on-device reference.

Source: benchmark_exynos2200.md (OceanguardAI-App memory, 2026-04-10).
GPU (Xclipse 920 AMD RDNA2) is not supported by either backend — CPU-only.
"""

# (backend, model, params, load_s, ttft_s, prefill_tps, decode_tps, note)
BENCHMARK_ROWS = [
    ("LiteRT-LM 0.10.0", "Gemma 4 E2B",   "2.3B eff", 1.1,  1.92, 58.1, 7.6,  "1.65x faster than llama.cpp (same model)"),
    ("llama.cpp",         "Qwen 3.5 0.8B", "0.8B",     3.2,  2.07, 95.5, 17.8, "Fastest decode, lower quality"),
    ("llama.cpp",         "Qwen 3.5 2B",   "2B",       6.1,  4.20, 47.2, 8.1,  "Best quality/speed tradeoff"),
    ("llama.cpp",         "Qwen 3.5 4B",   "4B",       19.3, 8.78, 22.6, 4.1,  "Highest quality"),
    ("llama.cpp",         "Gemma 4 E2B",   "2.3B eff", 8.8,  5.83, 33.5, 4.6,  "Short output (70 tok) — formatter investigation pending"),
]

TOKENS_PER_WORD = 1.3
REPORT_WORD_MIN, REPORT_WORD_MAX = 700, 900
_TOK_MIN = int(REPORT_WORD_MIN * TOKENS_PER_WORD)
_TOK_MAX = int(REPORT_WORD_MAX * TOKENS_PER_WORD)

_HDR_W = 108
print("VLM Benchmark — Samsung Galaxy S22 Ultra (Exynos 2200, CPU-only)")
print("=" * _HDR_W)
print(
    f"{'Backend':<20} {'Model':<16} {'Params':<10} {'Load':>6} {'TTFT':>6} "
    f"{'Prefill':>9} {'Decode':>8}  Note"
)
print("-" * _HDR_W)
for _backend, _model, _params, _load, _ttft, _pfill, _dec, _note in BENCHMARK_ROWS:
    print(
        f"{_backend:<20} {_model:<16} {_params:<10} {_load:>5.1f}s {_ttft:>5.2f}s "
        f"{_pfill:>7.1f}t/s {_dec:>6.1f}t/s  {_note}"
    )

print()
print(f"Expected latency for {REPORT_WORD_MIN}-{REPORT_WORD_MAX} word reports "
      f"({TOKENS_PER_WORD} tok/word -> {_TOK_MIN}-{_TOK_MAX} tokens):")
print()
print(f"{'Backend':<20} {'Model':<16} {'Min latency':>13} {'Max latency':>13}")
print("-" * 65)
for _backend, _model, _params, _load, _ttft, _pfill, _dec, _note in BENCHMARK_ROWS:
    _lat_min = round(_ttft + _TOK_MIN / _dec, 1)
    _lat_max = round(_ttft + _TOK_MAX / _dec, 1)
    print(f"{_backend:<20} {_model:<16} {_lat_min:>11.1f}s {_lat_max:>11.1f}s")

print()
print("Production choice (Android app): LiteRT-LM 0.10.0 + Gemma 4 E2B")
print(f"  TTFT 1.92s, decode 7.6 tok/s → expected "
      f"{round(1.92 + _TOK_MIN / 7.6, 0):.0f}-{round(1.92 + _TOK_MAX / 7.6, 0):.0f}s "
      f"for a {REPORT_WORD_MIN}-{REPORT_WORD_MAX} word report.")
print("  (Model load is cached after first call: 1.1s for subsequent runs.)")


## Submission Metadata

- **Track**: Global Resilience
- **License**: Apache 2.0
- **Repo**: https://github.com/asferrer/OceanguardAI
- **APK**: https://github.com/asferrer/OceanguardAI/releases/latest
- **Author**: Alejandro Sanchez Ferrer
